In [1]:
import pandas as pd
import numpy as np

Loading in the data

In [2]:
df_recipes = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/all_recipes.csv')
df_cuisines = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/cuisines.csv')

Check number of entries in each dataset
- no. of entries = no. of unique urls means 'url' can be used as a key
- much less entries in the cuisine dataset means we should only use the merged cuisine dataset for the first visualisation

In [3]:
print(f"Recipes data shape: {df_recipes.shape}")
print(f"Cuisines data shape: {df_cuisines.shape}")
print(f"Unique URLs in Recipes: {df_recipes['url'].nunique()}")
print(f"Unique URLs in Cuisines: {df_cuisines['url'].nunique()}")

Recipes data shape: (14426, 16)
Cuisines data shape: (2218, 17)
Unique URLs in Recipes: 14426
Unique URLs in Cuisines: 2218


Columns of concern are 'url', 'date_published', 'protein', 'calories', 'avg_rating', 'prep_time', 'name'.

Check for their type, as well as missing entries

In [4]:
print(df_recipes[['url', 'date_published', 'protein', 'calories', 'avg_rating', 'prep_time', 'name']].dtypes)
print(df_cuisines[['url', 'country']].dtypes)
critical_cols = ['url', 'date_published', 'protein', 'calories', 'avg_rating', 'prep_time', 'name']
print(df_recipes[critical_cols].isnull().sum())

url                object
date_published     object
protein           float64
calories          float64
avg_rating        float64
prep_time           int64
name               object
dtype: object
url        object
country    object
dtype: object
url                 0
date_published      0
protein           248
calories          200
avg_rating        972
prep_time           0
name                0
dtype: int64


Preview the data for the important columns.
- Protein value of 0 means that there will be protein/calorie ratio of 0
- Prep time of 0 might need to be filtered out
- Unusally high and low prep times may be problematic when visualising

In [5]:
df_subset = df_recipes[critical_cols]

print("Numeric data info")

numeric_summary = df_subset.select_dtypes(include='number').agg(['min', 'max'])
print(numeric_summary)


print("\nNon-numeric data info")
non_numeric_cols = df_subset.select_dtypes(exclude='number').columns
for col in non_numeric_cols:
    # Get total count of unique items
    unique_count = df_subset[col].nunique()
    
    # Get the actual text values, ignoring NaNs
    unique_values = df_subset[col].dropna().unique()
    
    print(f"\nColumn: '{col}'")
    print(f"Total Unique Values: {unique_count}")
    
    # Print only the first 10 values to keep the output clean
    print(f"Sample Values: {unique_values[:10]}")

Numeric data info
     protein  calories  avg_rating  prep_time
min      0.0       1.0         1.0          0
max    939.0    9538.0         5.0       2160

Non-numeric data info

Column: 'url'
Total Unique Values: 14426
Sample Values: ['https://www.allrecipes.com/recipe/140717/chewy-whole-wheat-peanut-butter-brownies/'
 'https://www.allrecipes.com/recipe/269204/pumpkin-pie-eggnog/'
 'https://www.allrecipes.com/recipe/238054/eggs-poached-in-tomato-sauce/'
 'https://www.allrecipes.com/minestrone-casserole-recipe-8765618'
 'https://www.allrecipes.com/recipe/241937/yummy-stuffed-peppers/'
 'https://www.allrecipes.com/recipe/219586/prime-rib-our-way/'
 'https://www.allrecipes.com/recipe/8636/parmesan-chicken-ii/'
 'https://www.allrecipes.com/recipe/12950/chicken-andouille-gumbo/'
 'https://www.allrecipes.com/recipe/185816/sweet-pork-for-burritos/'
 'https://www.allrecipes.com/recipe/239507/quick-baked-chicken-parmesan/']

Column: 'date_published'
Total Unique Values: 1542
Sample Values: ['

Check for number of unique cuisines

In [6]:
print(f"Sample of unique cuisines: {df_cuisines['country'].dropna().unique()[:10]}")

Sample of unique cuisines: ['Greek' 'Jewish' 'Australian and New Zealander' 'Chilean' 'Tex-Mex'
 'Canadian' 'Italian' 'Danish' 'Amish and Mennonite' 'Spanish']


Check for duplicate data, no duplicates found

In [7]:
print(f"Duplicate rows in Recipes: {df_recipes.duplicated().sum()}")
print(f"Duplicate rows in Cuisines: {df_cuisines.duplicated().sum()}")

Duplicate rows in Recipes: 0
Duplicate rows in Cuisines: 0


Since we are merging cuisines.csv with all_recipes.csv, check how many urls actually match.

Final merged df will have 1055 or less entries.

In [8]:
matches = df_cuisines['url'].isin(df_recipes['url']).sum()
total_cuisines = len(df_cuisines['url'])

print(f"Matching URLs: {matches} out of {total_cuisines}")

Matching URLs: 1055 out of 2218


Create a subset of all_recipes.csv with only the necessary columns and dropping all missing data. We are left with 13232/14426 entries.

In [9]:
df_base = df_subset.dropna(subset=critical_cols)
df_base.shape

(13232, 7)

Data type conversion:
- datetime for date
- category for country

In [10]:
df_base['date_published'] = pd.to_datetime(df_base['date_published'], format='%Y-%m-%d')
df_cuisines['country'] = df_cuisines['country'].astype('category')

C:\Users\seand\AppData\Local\Temp\ipykernel_5748\2858831330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base['date_published'] = pd.to_datetime(df_base['date_published'], format='%Y-%m-%d')


Create protein_calorie_ratio column

In [11]:
df_base['protein_calorie_ratio'] = df_base['protein'] / df_base['calories']

C:\Users\seand\AppData\Local\Temp\ipykernel_5748\3928749409.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base['protein_calorie_ratio'] = df_base['protein'] / df_base['calories']


Create merged df for 1st visualisation

In [12]:
df_merged = pd.merge(df_base, 
                    df_cuisines[['url', 'country']], 
                    on='url', 
                    how='inner')

df_merged = df_merged.dropna(subset=['country'])

Preview of the cleaned data
- df_base is before merging, for Viz 2 and Viz 3
- df_merged is after merging with country data, for Viz 1

In [13]:
print(f"Cleaned dataset: {df_base.shape}")
display(df_base.head())

print(f"\nCleaned dataset after merger: {df_merged.shape}")
display(df_merged.head())

Cleaned dataset: (13232, 8)


,url,date_published,protein,calories,avg_rating,prep_time,name,protein_calorie_ratio
0,https://www.allrecipes.com/recipe/140717/chewy...,2020-06-18,6.0,222.0,4.4,20,Chewy Whole Wheat Peanut Butter Brownies,0.027027
1,https://www.allrecipes.com/recipe/269204/pumpk...,2022-09-26,8.0,477.0,5.0,10,Pumpkin Pie Eggnog,0.016771
2,https://www.allrecipes.com/recipe/238054/eggs-...,2018-06-08,20.0,354.0,4.8,10,Eggs Poached in Tomato Sauce,0.056497
3,https://www.allrecipes.com/minestrone-casserol...,2025-03-03,19.0,356.0,4.3,20,Minestrone Casserole,0.053371
4,https://www.allrecipes.com/recipe/241937/yummy...,2024-12-11,19.0,366.0,4.7,30,Yummy Stuffed Peppers,0.051913



Cleaned dataset after merger: (1001, 9)


,url,date_published,protein,calories,avg_rating,prep_time,name,protein_calorie_ratio,country
0,https://www.allrecipes.com/recipe/282434/quinc...,2020-11-30,3.0,188.0,5.0,30,Quince Empanadas,0.015957,Argentinian
1,https://www.allrecipes.com/recipe/220128/chef-...,2024-01-19,66.0,1262.0,4.5,15,Chef John's Buttermilk Fried Chicken,0.052298,Soul Food
2,https://www.allrecipes.com/recipe/259179/polis...,2024-11-23,10.0,581.0,5.0,25,Polish Poppy Seed Cake,0.017212,Polish
3,https://www.allrecipes.com/recipe/45301/harose...,2025-03-20,4.0,241.0,4.8,20,Haroset for Passover,0.016598,Jewish
4,https://www.allrecipes.com/chicken-marsala-fet...,2024-03-11,57.0,861.0,4.6,15,Chicken Marsala Fettuccine,0.066202,Italian
